# Video Captioning with BLIP

Captions video frames using the BLIP image-captioning model, then burns the captions onto the frames and re-encodes a captioned video.

**Workflow:**
1. Get the video (local file or download from a link)
2. Cut the video into frames
3. Caption the frames with BLIP
4. Merge frames + captions back into a video

## Setup

Install dependencies (run once):
```bash
pip install torch transformers opencv-python requests yt-dlp
```

In [ ]:
import os
import re
import shutil
import tempfile
import urllib.parse
import urllib.request
from pathlib import Path

import cv2
import numpy as np
import requests
import torch
from PIL import Image, ImageDraw, ImageFont
from transformers import BlipForConditionalGeneration, BlipProcessor

print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())

## Configuration

In [ ]:
# @param {"type": "string"}
SOURCE = "test_video.mp4"  # local path, direct video URL, or page URL (e.g. YouTube)
# @param {"type": "string"}
OUTPUT_PATH = "captioned_video.mp4"
# @param {"type": "number", "step": 0.5}
FPS = 1.0               # frames per second to sample
MODEL_NAME = "Salesforce/blip-image-captioning-base"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"source={SOURCE!r}  output={OUTPUT_PATH!r}  fps={FPS}  device={DEVICE}")

## Step 1: Get the video

Downloads the video if `SOURCE` is a link (direct file link or a page like YouTube via yt-dlp), otherwise uses the local file.

In [ ]:
VIDEO_EXTS = {".mp4", ".mov", ".avi", ".mkv", ".webm", ".m4v"}

def is_url(source):
    return source.startswith("http://") or source.startswith("https://")

def looks_like_video_url(source):
    path = urllib.parse.urlparse(source).path.lower()
    return Path(path).suffix in VIDEO_EXTS

def download_with_ytdlp(url, dest_dir):
    import yt_dlp
    out_template = str(dest_dir / "source.%(ext)s")
    ydl_opts = {
        "outtmpl": out_template,
        "format": "bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]/best",
        "merge_output_format": "mp4",
        "quiet": True,
        "no_warnings": True,
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([url])
    downloaded = list(dest_dir.glob("source.*"))
    assert downloaded, "yt-dlp finished but no video file was produced."
    return downloaded[0]

def download_direct(url, dest_dir):
    filename = Path(urllib.parse.urlparse(url).path).name or "source.mp4"
    dest = dest_dir / filename
    with requests.get(url, stream=True, timeout=30) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        downloaded = 0
        with open(dest, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
                    downloaded += len(chunk)
                    if total:
                        print(f"\rDownloading: {downloaded * 100 // total}%", end="", flush=True)
    print()
    return dest

work_dir = Path(tempfile.mkdtemp(prefix="video_caption_"))

if not is_url(SOURCE):
    assert Path(SOURCE).is_file(), f"Video file not found: {SOURCE}"
    video_path = Path(SOURCE)
    print(f"[1/4] Using local video: {video_path}")
elif looks_like_video_url(SOURCE):
    print(f"[1/4] Downloading direct file: {SOURCE}")
    video_path = download_direct(SOURCE, work_dir)
else:
    print(f"[1/4] Downloading via yt-dlp: {SOURCE}")
    video_path = download_with_ytdlp(SOURCE, work_dir)

video_path

## Step 2: Cut the video into frames

In [ ]:
frames_dir = work_dir / "frames"
frames_dir.mkdir(parents=True, exist_ok=True)

cap = cv2.VideoCapture(str(video_path))
assert cap.isOpened(), f"Could not open video: {video_path}"

video_fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"[2/4] Video: {width}x{height} @ {video_fps:.2f} fps, {total_frames} frames")

frame_paths = []
frame_idx = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break
    if frame_idx % max(1, round(video_fps / FPS)) == 0:
        frame_path = frames_dir / f"frame_{len(frame_paths):06d}.jpg"
        cv2.imwrite(str(frame_path), frame)
        frame_paths.append(frame_path)
    frame_idx += 1
cap.release()
print(f"      Extracted {len(frame_paths)} frames")
frame_paths[:3]

## Step 3: Caption the frames with BLIP

First run downloads the model (~1 GB) from Hugging Face.

In [ ]:
print(f"[3/4] Loading BLIP model: {MODEL_NAME} (device: {DEVICE})")
processor = BlipProcessor.from_pretrained(MODEL_NAME)
model = BlipForConditionalGeneration.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()

In [ ]:
captions = []
for i, frame_path in enumerate(frame_paths, 1):
    image = Image.open(frame_path).convert("RGB")
    inputs = processor(images=image, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=50)
    caption = processor.decode(output[0], skip_special_tokens=True)
    captions.append(caption)
    print(f"[{i}/{len(frame_paths)}] {caption}")

captions[:5]

## Step 4: Merge frames + captions back into a video

In [ ]:
def find_font(size):
    for name in ["arial.ttf", "segoeui.ttf", "calibri.ttf", "DejaVuSans.ttf"]:
        try:
            return ImageFont.truetype(name, size)
        except OSError:
            continue
    return ImageFont.load_default()

def burn_caption(frame, caption):
    """Draw the caption onto the bottom of the frame."""
    img = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    draw = ImageDraw.Draw(img)
    w, h = img.size

    font_size = max(16, w // 40)
    font = find_font(font_size)

    # Word-wrap the caption to the frame width
    words = caption.split()
    lines, line = [], ""
    for word in words:
        trial = f"{line} {word}".strip()
        if draw.textbbox((0, 0), trial, font=font)[2] <= w - 20:
            line = trial
        else:
            if line:
                lines.append(line)
            line = word
    if line:
        lines.append(line)

    line_height = font_size + 6
    box_height = line_height * len(lines) + 16
    overlay = Image.new("RGBA", img.size, (0, 0, 0, 0))
    ImageDraw.Draw(overlay).rectangle([(0, h - box_height), (w, h)], fill=(0, 0, 0, 160))
    img = Image.alpha_composite(img.convert("RGBA"), overlay).convert("RGB")
    draw = ImageDraw.Draw(img)

    y = h - box_height + 8
    for ln in lines:
        bbox = draw.textbbox((0, 0), ln, font=font)
        x = (w - (bbox[2] - bbox[0])) // 2
        draw.text((x, y), ln, font=font, fill=(255, 255, 255))
        y += line_height

    return cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)

In [ ]:
output_path = Path(OUTPUT_PATH)
output_path.parent.mkdir(parents=True, exist_ok=True)

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(str(output_path), fourcc, FPS, (width, height))
assert writer.isOpened(), f"Could not create output video: {output_path}"

print(f"[4/4] Writing captioned video to {output_path}")
for frame_path, caption in zip(frame_paths, captions):
    frame = cv2.imread(str(frame_path))
    writer.write(burn_caption(frame, caption))
writer.release()

# Timestamped caption log
sidecar = output_path.with_suffix(".txt")
with open(sidecar, "w", encoding="utf-8") as f:
    for i, cap in enumerate(captions):
        f.write(f"{i / FPS:8.2f}s  {cap}\n")

print(f"Done! {output_path}  ({output_path.stat().st_size / 1e6:.1f} MB)")
print(sidecar.read_text(encoding="utf-8"))

## Preview a captioned frame

In [ ]:
from IPython.display import Image as IPImage, display

sample = burn_caption(cv2.imread(str(frame_paths[0])), captions[0])
cv2.imwrite(str(work_dir / "preview.jpg"), sample)
display(IPImage(filename=str(work_dir / "preview.jpg")))

In [ ]:
# Clean up the temporary working directory when done
shutil.rmtree(work_dir, ignore_errors=True)
print("cleaned up")